# General Overview of making OPENMC models of Nuclear Reactors

## Used for Cardinal Multiphysics simulation

### Goal

Build a Basic Understanding of how to use the OPENMC Python API. This tutorial uses the example of creating a basic Mono-block HTGR core.

### Learning Objectives

&nbsp;1. Understand how to setup OPENMC Python API

&nbsp;2. Learn how to define OPENMC materials and properties in python

&nbsp;3. Learn how to define OPENMC geometries in python

&nbsp;4. Learn how to export those files into formats usable by further tools

# Setup
**Note:** This tutorial assumes you have a built OPENMC binary, someway, somehow. 
## Packages
We need to ensure any needed packages are all set for python. First, run the following code section, you can ignore anything about a version error

In [ ]:

!python -m pip install pip-review
!pip-review --auto
!python -m pip install openmc-plotter
import openmc


  Using cached pyvista_validation-0.5.4-cp310-abi3-manylinux2014_aarch64.manylinux_2_17_aarch64.manylinux_2_28_aarch64.whl.metadata (8.7 kB)
  Using cached trame_common-1.3.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached trame_rca-2.12.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached trame_server-4.1.2-py3-none-any.whl.metadata (5.8 kB)
  Attempting uninstall: trame-common
    Found existing installation: trame-common 1.2.9
    Uninstalling trame-common-1.2.9:
      Successfully uninstalled trame-common-1.2.9
  Attempting uninstall: pyvista-validation
    Found existing installation: pyvista-validation 0.2.2
    Uninstalling pyvista-validation-0.2.2:
      Successfully uninstalled pyvista-validation-0.2.2
  Attempting uninstall: trame-server
    Found existing installation: trame-server 4.1.0
    Uninstalling trame-server-4.1.0:
      Successfully uninstalled trame-server-4.1.0
  Attempting uninstall: trame-rca
    Found existing installation: trame-rca 2.11.8
    Uninstalling t

If that code block runs without errors, you can keep going. If it does not, execute the following code block. It will gather the OpenMC Python Module

*Note: Whenever OpenMC is updated by cardinal in the future, it will require rebuilding the module, which you can do the short way with this code cell*

In [36]:

%%bash
cd ${NEKRS_HOME%/*}/contrib/openmc
git fetch --tags
python -m pip install setuptools
python -m pip install numpy
python -m pip install .
python -c "import openmc; import openmc.lib; print(openmc.__version__); print(openmc.lib)"



Processing ./.
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for openmc: filename=openmc-0.16.1.dev14+g150f9ac04-py3-none-any.whl size=32327931 sha256=2797398118a938946390680698d77610e95a5fb31326070fcd2d609691dd0bbe
  Stored in directory: /tmp/pip-ephem-wheel-cache-gcajr00d/wheels/c5/3a/e3/a348133f2d8a151e8af8834edf742868ba29d162498eff47da
Successfully built openmc
  Attempting uninstall: openmc
    Found existing installation: openmc 0.16.1.dev14+g150f9ac04
    Uninstalling openmc-0.16.1.dev14+g150f9ac04:
      Successfully uninstalled openmc-0.16.1.dev14+g150f9ac04
0.16.1.dev14+g150f9ac04
<module 'openmc.lib' from '/home/jmiller/cardinal/contrib/openmc/openmc/lib/__init__.py'>


# Functions
Any neccessary imports and other setup.

In [35]:
import openmc
import numpy as np
import os

# 1. OpenMC Basic Tenants

## &nbsp;A. Structure


&nbsp;What goes into an OpenMC neutronics model?

### &nbsp;1. Geometry

&nbsp;&nbsp;&nbsp;Pretty Self-explanatory. Basically the actual structure of everything. OpenMC uses **CSG Models**, which is just a different way to go about things that a mesh. 
&nbsp;&nbsp;&nbsp;Basically , in CSG, to store a cube with a hole in it, it would be stored as a cube minus a cylinder, whereas a mesh is just the set of points that make up that cube with 
&nbsp;&nbsp;&nbsp;a hole in it

### &nbsp;2. Initial Conditions

&nbsp;&nbsp;&nbsp;Another Very straight forward component. Things like intial temperature, Values of any intial neutron sources being emitted into the problem.